In [31]:
import pandas as pd
import sqlite3

lib_db = sqlite3.connect('download.db')
book_catalog = pd.read_json('download.json')
kickoff_signups = pd .read_html('download.html', flavor='lxml')[0]
kickoff_signups.columns = (
    kickoff_signups.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)


In [11]:
# How much is each member borrowing?
# A per-member count of checkouts — including members who haven't borrowed anything at all.
q1 = """
SELECT 
    m.member_id, 
    m.first_name, 
    m.last_name, 
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC;
"""
print(pd.read_sql_query(q1, lib_db))

    member_id first_name last_name  total_checkouts
0        1034        Aya     Wahba               25
1        1044     Sherif     Saleh               21
2        1008       Ziad     Saleh               19
3        1010       Nour     Nabil               18
4        1027    Mostafa     Fouad               18
..        ...        ...       ...              ...
75       1063      Layla     Fouad                0
76       1064      Fares     Sabry                0
77       1066       Amir     Wahba                0
78       1069     Bassel      Adel                0
79       1078     Habiba     Osman                0

[80 rows x 4 columns]


In [13]:
# Which books match a chosen author pattern?
# A search across the catalog based on a letter or pattern you choose and record.
q2 = """
SELECT 
    book_id,
    title, 
    author
FROM books
WHERE author LIKE '%Samir%';
"""
print(pd.read_sql_query(q2, lib_db))

   book_id                  title       author
0      531  Voices in the Library  Samir Zohdy
1      532      The Last Bookmark  Samir Zohdy


In [14]:
# What are the most popular books?
# The five most-borrowed titles, ranked by how many times each was checked out.
q3 = """
SELECT 
    b.book_id, 
    b.title, 
    COUNT(c.checkout_id) AS checkout_count
FROM books b
JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""
print(pd.read_sql_query(q3, lib_db))

   book_id                   title  checkout_count
0      501         The Silver Kite              57
1      507   Fossils and Fireflies              55
2      513  Circuits for Beginners              46
3      519        Kites Over Cairo              38
4      525    Storms and Sailboats              25


In [15]:
# Who are the most active readers?
# The ten members who've borrowed the most books, ranked from highest to lowest.
q4 = """
SELECT 
    m.member_id, 
    m.first_name, 
    m.last_name, 
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC
LIMIT 10;
"""
print(pd.read_sql_query(q4, lib_db))

   member_id first_name last_name  total_checkouts
0       1034        Aya     Wahba               25
1       1044     Sherif     Saleh               21
2       1008       Ziad     Saleh               19
3       1010       Nour     Nabil               18
4       1027    Mostafa     Fouad               18
5       1018      Ahmed    Shafik               17
6       1024    Youssef    Hegazy               17
7       1065       Adam     Fahmy               17
8       1030       Reem     Osman               16
9       1047       Sara    Rashad               16


In [16]:
# What does a neighborhood's activity look like further back in time?
# Checkouts from one neighborhood you choose, ordered newest to oldest, looking past the ten most recent.
q5 = """
SELECT 
    c.checkout_id, 
    c.member_id, 
    c.book_id, 
    c.checkout_date, 
    m.neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Zamalek'
ORDER BY c.checkout_date DESC
LIMIT -1 OFFSET 10;
"""
print(pd.read_sql_query(q5, lib_db))

    checkout_id  member_id  book_id checkout_date neighborhood
0          9318       1067      513    2025-07-23      Zamalek
1          9320       1073      501    2025-07-19      Zamalek
2          9311       1065      507    2025-06-26      Zamalek
3          9298       1065      507    2025-06-18      Zamalek
4          9307       1072      501    2025-06-10      Zamalek
5          9333       1072      520    2025-06-03      Zamalek
6          9342       1068      501    2025-05-28      Zamalek
7          9297       1065      501    2025-04-14      Zamalek
8          9330       1072      513    2025-04-11      Zamalek
9          9288       1065      519    2025-03-20      Zamalek
10         9327       1065      513    2025-03-09      Zamalek
11         9329       1062      513    2025-03-06      Zamalek
12         9285       1071      513    2025-02-24      Zamalek
13         9341       1065      516    2025-02-19      Zamalek
14         9331       1065      525    2025-02-17      

In [32]:
df_members = pd.read_sql_query("SELECT * FROM members", lib_db)
df_checkouts = pd.read_sql_query("SELECT * FROM checkouts", lib_db)
df_books_sql = pd.read_sql_query("SELECT * FROM books", lib_db)

df_st1 = pd.merge(df_checkouts, df_members, on='member_id', how='left')
total_member_checkouts = df_checkouts.groupby('member_id').size().reset_index(name='total_member_checkouts')
df_st1 = pd.merge(df_st1, total_member_checkouts, on='member_id', how='left')
print(df_st1[['member_id', 'first_name', 'last_name', 'total_member_checkouts']])

     member_id first_name last_name  total_member_checkouts
0         1047       Sara    Rashad                      16
1         1072       Seif      Zaki                      14
2         1053       Adam    Shafik                       5
3         1032       Nada      Zaki                       6
4         1079       Rana     Osman                      10
..         ...        ...       ...                     ...
386       1044     Sherif     Saleh                      21
387       1008       Ziad     Saleh                      19
388       1024    Youssef    Hegazy                      17
389       1076       Dina     Wahba                       7
390       1044     Sherif     Saleh                      21

[391 rows x 4 columns]


In [33]:
df_all_books = pd.merge(df_books_sql, book_catalog, on='book_id', how='left')
df_st2 = pd.merge(df_st1, df_all_books, on='book_id', how='left')
df_st2['source'] = 'Database'
print(df_st2)

     checkout_id  member_id  book_id checkout_date return_date first_name  \
0           9263       1047      517    2024-10-21  2024-11-07       Sara   
1           9340       1072      513    2025-08-24  2025-09-01       Seif   
2           9231       1053      523    2024-02-04  2024-02-16       Adam   
3           9129       1032      513    2025-06-21  2025-06-29       Nada   
4           9370       1079      511    2025-11-11  2025-12-03       Rana   
..           ...        ...      ...           ...         ...        ...   
386         9232       1044      513    2025-05-26  2025-06-11     Sherif   
387         9084       1008      511    2024-06-27  2024-07-22       Ziad   
388         9116       1024      519    2025-01-10  2025-02-09    Youssef   
389         9352       1076      501    2025-02-02        None       Dina   
390         9246       1044      529    2025-06-25  2025-07-04     Sherif   

    last_name  grade neighborhood membership_status   join_date  \
0      R

In [35]:
kickoff_signups['checkout_id'] = None
kickoff_signups['return_date'] = None
kickoff_signups['source'] = 'Reading Kickoff Web Page'
df_kickoff = pd.merge(kickoff_signups, df_members, on='member_id', how='left')
df_kickoff = pd.merge(df_kickoff, df_all_books, on='book_id', how='left')
df_kickoff = pd.merge(df_kickoff, total_member_checkouts, on='member_id', how='left')
df_compainde= pd.concat([df_st2, df_kickoff], ignore_index=True)
df_compainde

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_member_checkouts,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16.0,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5.0,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Database
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Database
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10.0,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,Database
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,None,1003,501,2025-07-08,None,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9.0,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Reading Kickoff Web Page
413,None,1017,507,2025-07-11,None,Adam,Badr,9.0,Maadi,Active,2025-07-12,2.0,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press,Reading Kickoff Web Page
414,None,1061,504,2025-07-06,None,Ziad,Fahmy,9.0,zamalek,Inactive,2023-04-27,8.0,Rooftop Astronomers,Adel Roushdy,Science,319,2009.0,Cairo Young Readers,Reading Kickoff Web Page
415,None,1201,523,2025-07-08,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Reading Kickoff Web Page


In [37]:
df_compainde.to_csv('cmbained_data.csv', index=False)